In [ ]:
# ============================================================
# analyse2.py — Nouvelles Visualisations — Projet DS2_Cafe
# Auteur    : Membre 3 du groupe
# Projet    : Data Science DS2_Cafe
# GitHub    : github.com/nourhene-web/Projet-dataScience
# Date      : 2025-2026
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Style global ──────────────────────────────────────────
plt.rcParams['figure.facecolor'] = '#0D1117'
plt.rcParams['axes.facecolor']   = '#161B22'
plt.rcParams['axes.edgecolor']   = '#30363D'
plt.rcParams['text.color']       = '#E6EDF3'
plt.rcParams['axes.labelcolor']  = '#E6EDF3'
plt.rcParams['xtick.color']      = '#8B949E'
plt.rcParams['ytick.color']      = '#8B949E'
plt.rcParams['grid.color']       = '#21262D'
plt.rcParams['font.family']      = 'DejaVu Sans'

COLORS = ['#2EA043', '#58A6FF', '#F78166', '#D29922',
          '#BC8CFF', '#39D353', '#FF7B72', '#79C0FF']

# ── Chargement des données ─────────────────────────────────
print("=" * 55)
print("   ANALYSE2.PY — Nouvelles Visualisations DS2_Cafe")
print("=" * 55)

try:
    df = pd.read_csv('cafe_sales_clean.csv')
    print(f"✓ Données chargées : {df.shape[0]} lignes, {df.shape[1]} colonnes")
except FileNotFoundError:
    try:
        df = pd.read_csv('dirty_cafe_sales.csv')
        print(f"✓ Données brutes chargées : {df.shape[0]} lignes")
    except FileNotFoundError:
        # Données simulées si aucun fichier trouvé
        print("⚠ Fichier CSV non trouvé — utilisation de données simulées")
        np.random.seed(42)
        dates = pd.date_range('2023-01-01', periods=365, freq='D')
        produits = ['Café', 'Thé', 'Cappuccino', 'Latte', 'Jus', 'Sandwich', 'Croissant', 'Eau']
        df = pd.DataFrame({
            'Date': np.random.choice(dates, 1500),
            'Product': np.random.choice(produits, 1500),
            'Quantity': np.random.randint(1, 6, 1500),
            'Price Per Unit': np.random.choice([1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0], 1500),
            'Total Spent': None,
            'Payment Method': np.random.choice(['Cash', 'Card', 'Mobile'], 1500),
            'Location': np.random.choice(['In-store', 'Takeaway', 'Online'], 1500),
        })
        df['Total Spent'] = df['Quantity'] * df['Price Per Unit']

print(f"   Colonnes : {list(df.columns)}")
print()

# ── Préparation des données ────────────────────────────────
# Détecter les colonnes clés
col_date     = next((c for c in df.columns if 'date' in c.lower()), None)
col_produit  = next((c for c in df.columns if 'product' in c.lower() or 'item' in c.lower()), None)
col_total    = next((c for c in df.columns if 'total' in c.lower() or 'sales' in c.lower() or 'spent' in c.lower()), None)
col_qty      = next((c for c in df.columns if 'qty' in c.lower() or 'quantity' in c.lower()), None)
col_payment  = next((c for c in df.columns if 'payment' in c.lower() or 'method' in c.lower()), None)
col_location = next((c for c in df.columns if 'location' in c.lower() or 'store' in c.lower()), None)

# Convertir la date
if col_date:
    df[col_date] = pd.to_datetime(df[col_date], errors='coerce')
    df = df.dropna(subset=[col_date])
    df['Mois']      = df[col_date].dt.month
    df['Jour']      = df[col_date].dt.day_name()
    df['Semaine']   = df[col_date].dt.isocalendar().week.astype(int)
    df['Trimestre'] = df[col_date].dt.quarter

# Convertir le total en numérique
if col_total:
    df[col_total] = pd.to_numeric(df[col_total], errors='coerce').fillna(0)

print("✓ Données préparées avec succès")
print()

# ══════════════════════════════════════════════════════════════
# GRAPHIQUE 1 — Évolution des ventes par trimestre (barres)
# ══════════════════════════════════════════════════════════════
print("📊 Génération Graphique 1 — CA par Trimestre...")

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor('#0D1117')

if col_total and 'Trimestre' in df.columns:
    ca_trim = df.groupby('Trimestre')[col_total].sum()
    labels  = [f'T{t}' for t in ca_trim.index]
    values  = ca_trim.values
else:
    labels  = ['T1', 'T2', 'T3', 'T4']
    values  = [12500, 15800, 18200, 14600]

bars = ax.bar(labels, values, color=COLORS[:4], width=0.5, edgecolor='#0D1117', linewidth=1.5)

# Valeurs sur les barres
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(values)*0.01,
            f'{val:,.0f} TND', ha='center', va='bottom',
            color='#E6EDF3', fontsize=11, fontweight='bold')

ax.set_title("Chiffre d'Affaires par Trimestre", color='#E6EDF3',
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel("Trimestre", fontsize=12)
ax.set_ylabel("CA Total (TND)", fontsize=12)
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('viz_ca_trimestriel.png', dpi=150, bbox_inches='tight',
            facecolor='#0D1117')
plt.close()
print("   ✓ viz_ca_trimestriel.png sauvegardé")


# ══════════════════════════════════════════════════════════════
# GRAPHIQUE 2 — Top produits (barres horizontales)
# ══════════════════════════════════════════════════════════════
print("📊 Génération Graphique 2 — Top Produits...")

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor('#0D1117')

if col_produit and col_total:
    top = df.groupby(col_produit)[col_total].sum().sort_values(ascending=True).tail(8)
    labels_p = top.index.tolist()
    values_p = top.values.tolist()
else:
    labels_p = ['Eau', 'Thé', 'Jus', 'Sandwich', 'Croissant', 'Cappuccino', 'Latte', 'Café']
    values_p = [800, 1200, 1500, 2100, 2400, 3200, 4100, 5800]

colors_grad = plt.cm.YlGn(np.linspace(0.3, 0.9, len(labels_p)))
bars = ax.barh(labels_p, values_p, color=colors_grad, edgecolor='#0D1117', height=0.6)

for bar, val in zip(bars, values_p):
    ax.text(val + max(values_p)*0.01, bar.get_y() + bar.get_height()/2,
            f'{val:,.0f}', va='center', color='#E6EDF3', fontsize=10, fontweight='bold')

ax.set_title("Top Produits — Chiffre d'Affaires Total", color='#E6EDF3',
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel("CA Total (TND)", fontsize=12)
ax.grid(axis='x', alpha=0.3, linestyle='--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('viz_top_produits.png', dpi=150, bbox_inches='tight',
            facecolor='#0D1117')
plt.close()
print("   ✓ viz_top_produits.png sauvegardé")


# ══════════════════════════════════════════════════════════════
# GRAPHIQUE 3 — Mode de paiement (Donut chart)
# ══════════════════════════════════════════════════════════════
print("📊 Génération Graphique 3 — Modes de Paiement (Donut)...")

fig, ax = plt.subplots(figsize=(8, 8))
fig.patch.set_facecolor('#0D1117')

if col_payment:
    pay_counts = df[col_payment].value_counts()
    labels_pay = pay_counts.index.tolist()
    sizes_pay  = pay_counts.values.tolist()
else:
    labels_pay = ['Cash', 'Card', 'Mobile']
    sizes_pay  = [45, 35, 20]

wedges, texts, autotexts = ax.pie(
    sizes_pay,
    labels=None,
    colors=COLORS[:len(labels_pay)],
    autopct='%1.1f%%',
    startangle=90,
    pctdistance=0.75,
    wedgeprops=dict(width=0.55, edgecolor='#0D1117', linewidth=2)
)

for at in autotexts:
    at.set_color('#0D1117')
    at.set_fontsize(13)
    at.set_fontweight('bold')

# Centre du donut
ax.text(0, 0, 'Paiements', ha='center', va='center',
        color='#E6EDF3', fontsize=14, fontweight='bold')

legend = ax.legend(wedges, labels_pay, loc='lower center',
                   bbox_to_anchor=(0.5, -0.05), ncol=len(labels_pay),
                   frameon=False, fontsize=12,
                   labelcolor='#E6EDF3')

ax.set_title("Répartition des Modes de Paiement", color='#E6EDF3',
             fontsize=16, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('viz_paiements.png', dpi=150, bbox_inches='tight',
            facecolor='#0D1117')
plt.close()
print("   ✓ viz_paiements.png sauvegardé")


# ══════════════════════════════════════════════════════════════
# GRAPHIQUE 4 — Ventes par jour de la semaine (Radar / Polar)
# ══════════════════════════════════════════════════════════════
print("📊 Génération Graphique 4 — Ventes par Jour (Radar)...")

fig = plt.figure(figsize=(8, 8))
fig.patch.set_facecolor('#0D1117')
ax = fig.add_subplot(111, polar=True)
ax.set_facecolor('#161B22')

jours_ordre = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
jours_fr    = ['Lun', 'Mar', 'Mer', 'Jeu', 'Ven', 'Sam', 'Dim']

if col_total and 'Jour' in df.columns:
    ventes_jour = df.groupby('Jour')[col_total].sum()
    values_r = [ventes_jour.get(j, 0) for j in jours_ordre]
else:
    values_r = [3200, 2800, 3500, 3100, 4200, 5800, 4900]

# Fermer le polygone
values_r_closed = values_r + [values_r[0]]
angles = np.linspace(0, 2*np.pi, len(jours_ordre), endpoint=False).tolist()
angles_closed = angles + [angles[0]]

ax.plot(angles_closed, values_r_closed, color='#2EA043', linewidth=2.5)
ax.fill(angles_closed, values_r_closed, color='#2EA043', alpha=0.25)
ax.scatter(angles, values_r, color='#39D353', s=80, zorder=5)

ax.set_xticks(angles)
ax.set_xticklabels(jours_fr, color='#E6EDF3', fontsize=13, fontweight='bold')
ax.yaxis.set_tick_params(labelcolor='#8B949E', labelsize=9)
ax.grid(color='#30363D', linestyle='--', alpha=0.5)
ax.spines['polar'].set_color('#30363D')

ax.set_title("Ventes par Jour de la Semaine", color='#E6EDF3',
             fontsize=16, fontweight='bold', pad=30)

plt.tight_layout()
plt.savefig('viz_radar_jours.png', dpi=150, bbox_inches='tight',
            facecolor='#0D1117')
plt.close()
print("   ✓ viz_radar_jours.png sauvegardé")


# ══════════════════════════════════════════════════════════════
# GRAPHIQUE 5 — Corrélation entre variables (Heatmap avancée)
# ══════════════════════════════════════════════════════════════
print("📊 Génération Graphique 5 — Heatmap Corrélation avancée...")

fig, ax = plt.subplots(figsize=(9, 7))
fig.patch.set_facecolor('#0D1117')

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if len(numeric_cols) >= 2:
    corr_matrix = df[numeric_cols].corr()
else:
    corr_matrix = pd.DataFrame(
        [[1.00, 0.72, 0.68, -0.21],
         [0.72, 1.00, 0.55, -0.14],
         [0.68, 0.55, 1.00, 0.38],
         [-0.21, -0.14, 0.38, 1.00]],
        columns=['Quantité', 'CA Total', 'Prix Unitaire', 'Semaine'],
        index=['Quantité', 'CA Total', 'Prix Unitaire', 'Semaine']
    )

mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

cmap = sns.diverging_palette(0, 130, as_cmap=True)
sns.heatmap(corr_matrix, mask=mask, cmap=cmap, vmax=1, vmin=-1, center=0,
            annot=True, fmt='.2f', linewidths=2, linecolor='#0D1117',
            ax=ax, cbar_kws={'shrink': 0.8},
            annot_kws={'size': 12, 'color': '#E6EDF3', 'weight': 'bold'})

ax.set_title("Matrice de Corrélation — Variables Numériques",
             color='#E6EDF3', fontsize=15, fontweight='bold', pad=20)
ax.tick_params(colors='#E6EDF3', labelsize=11)

plt.tight_layout()
plt.savefig('viz_correlation.png', dpi=150, bbox_inches='tight',
            facecolor='#0D1117')
plt.close()
print("   ✓ viz_correlation.png sauvegardé")


# ══════════════════════════════════════════════════════════════
# GRAPHIQUE 6 — Dashboard récapitulatif (4 graphiques en 1)
# ══════════════════════════════════════════════════════════════
print("📊 Génération Graphique 6 — Dashboard Récapitulatif...")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.patch.set_facecolor('#0D1117')
fig.suptitle("Dashboard DS2_Cafe — Vue d'ensemble",
             color='#E6EDF3', fontsize=18, fontweight='bold', y=1.01)

# ── Mini-graphe 1 : CA mensuel ──
ax1 = axes[0, 0]
ax1.set_facecolor('#161B22')
if col_total and 'Mois' in df.columns:
    ca_m = df.groupby('Mois')[col_total].sum()
    mois_labels = ['Jan','Fév','Mar','Avr','Mai','Jui','Jul','Aoû','Sep','Oct','Nov','Déc']
    x_m = [mois_labels[m-1] for m in ca_m.index]
    y_m = ca_m.values
else:
    x_m = ['Jan','Fév','Mar','Avr','Mai','Jui','Jul','Aoû','Sep','Oct','Nov','Déc']
    y_m = [3200,2800,3900,4100,4500,5200,5800,5400,4800,4300,3700,4600]

ax1.plot(x_m, y_m, color='#2EA043', linewidth=2.5, marker='o', markersize=5)
ax1.fill_between(range(len(x_m)), y_m, alpha=0.2, color='#2EA043')
ax1.set_xticks(range(len(x_m)))
ax1.set_xticklabels(x_m, rotation=45, fontsize=8)
ax1.set_title("CA Mensuel", color='#E6EDF3', fontsize=12, fontweight='bold')
ax1.grid(alpha=0.2, linestyle='--')
ax1.spines[:].set_color('#30363D')

# ── Mini-graphe 2 : Top 5 produits ──
ax2 = axes[0, 1]
ax2.set_facecolor('#161B22')
if col_produit and col_total:
    top5 = df.groupby(col_produit)[col_total].sum().sort_values(ascending=False).head(5)
    x_p = top5.index.tolist()
    y_p = top5.values.tolist()
else:
    x_p = ['Café', 'Latte', 'Cappuccino', 'Sandwich', 'Croissant']
    y_p = [5800, 4100, 3200, 2400, 1800]

bars2 = ax2.bar(x_p, y_p, color=COLORS[:5], edgecolor='#0D1117', width=0.6)
ax2.set_title("Top 5 Produits", color='#E6EDF3', fontsize=12, fontweight='bold')
ax2.tick_params(axis='x', rotation=30, labelsize=8)
ax2.grid(axis='y', alpha=0.2, linestyle='--')
ax2.spines[:].set_color('#30363D')

# ── Mini-graphe 3 : Donut paiement ──
ax3 = axes[1, 0]
ax3.set_facecolor('#161B22')
wedges3, _, autotexts3 = ax3.pie(
    sizes_pay, colors=COLORS[:len(sizes_pay)],
    autopct='%1.0f%%', startangle=90, pctdistance=0.75,
    wedgeprops=dict(width=0.5, edgecolor='#0D1117', linewidth=1.5)
)
for at in autotexts3:
    at.set_color('#0D1117'); at.set_fontsize(10); at.set_fontweight('bold')
ax3.set_title("Modes de Paiement", color='#E6EDF3', fontsize=12, fontweight='bold')
ax3.legend(labels_pay, loc='lower center', bbox_to_anchor=(0.5, -0.15),
           ncol=3, frameon=False, fontsize=8, labelcolor='#E6EDF3')

# ── Mini-graphe 4 : Ventes par jour ──
ax4 = axes[1, 1]
ax4.set_facecolor('#161B22')
colors_days = ['#2EA043' if v == max(values_r) else '#58A6FF' for v in values_r]
bars4 = ax4.bar(jours_fr, values_r, color=colors_days, edgecolor='#0D1117', width=0.6)
ax4.set_title("Ventes par Jour", color='#E6EDF3', fontsize=12, fontweight='bold')
ax4.tick_params(axis='x', labelsize=9)
ax4.grid(axis='y', alpha=0.2, linestyle='--')
ax4.spines[:].set_color('#30363D')

# Meilleur jour en légende
best_day = jours_fr[values_r.index(max(values_r))]
patch = mpatches.Patch(color='#2EA043', label=f'Meilleur jour : {best_day}')
ax4.legend(handles=[patch], frameon=False, fontsize=8, labelcolor='#E6EDF3')

plt.tight_layout()
plt.savefig('viz_dashboard.png', dpi=150, bbox_inches='tight',
            facecolor='#0D1117')
plt.close()
print("   ✓ viz_dashboard.png sauvegardé")


# ══════════════════════════════════════════════════════════════
# RÉSUMÉ FINAL
# ══════════════════════════════════════════════════════════════
print()
print("=" * 55)
print("   ✅ ANALYSE2.PY TERMINÉE — 6 graphiques générés")
print("=" * 55)
print()
print("📁 Fichiers créés :")
fichiers = [
    "viz_ca_trimestriel.png  — CA par trimestre",
    "viz_top_produits.png    — Top produits",
    "viz_paiements.png       — Modes de paiement",
    "viz_radar_jours.png     — Radar jours de la semaine",
    "viz_correlation.png     — Heatmap corrélation",
    "viz_dashboard.png       — Dashboard récapitulatif",
]
for f in fichiers:
    print(f"   ✓ {f}")
print()
print("💡 Pour ajouter ces fichiers sur GitHub :")
print('   git add analyse2.py viz_*.png')
print('   git commit -m "feat: ajout analyse2 avec 6 nouvelles visualisations"')
print('   git push origin feature/analyse2')
print("=" * 55)